# 🚀 BCTC & ANNUAL REPORT 5-WORKER DISTRIBUTED DATA LAKE PIPELINE
### Bóc tách toàn bộ BCTC (Quý, Năm, TTM) và Báo cáo thường niên của ~1.300 mã cổ phiếu

Notebook này được thiết kế để chạy song song trên **5 tài khoản Google Colab miễn phí**:
- Mỗi tài khoản phụ trách **~260 mã** cổ phiếu.
- **Stream & Purge**: Tải PDF vào đĩa tạm 100GB của Colab $\rightarrow$ Bóc tách số liệu kế toán (B01, B02, B03, TTM) và Kế hoạch năm tới từ BCTN $\rightarrow$ **Xoá file PDF ngay** để không làm đầy Google Drive.
- **Incremental Sync & Checkpoint**: Tự động kế thừa dữ liệu đã cache từ trước (trong `extracted_bctc_lake.json` hoặc shard cũ), chỉ cào những mã còn thiếu, tự động lưu tiến độ sau mỗi 5 mã vào `bctc_shard_{WORKER_ID}.json`.

> **HƯỚNG DẪN CHẠY TRÊN 5 TÀI KHOẢN**:
> - **Tài khoản 1**: Đặt `WORKER_ID = 0` $\rightarrow$ Bấm **Runtime** -> **Run all**
> - **Tài khoản 2**: Đặt `WORKER_ID = 1` $\rightarrow$ Bấm **Runtime** -> **Run all**
> - **Tài khoản 3**: Đặt `WORKER_ID = 2` $\rightarrow$ Bấm **Runtime** -> **Run all**
> - **Tài khoản 4**: Đặt `WORKER_ID = 3` $\rightarrow$ Bấm **Runtime** -> **Run all**
> - **Tài khoản 5**: Đặt `WORKER_ID = 4` $\rightarrow$ Bấm **Runtime** -> **Run all**

In [ ]:
# =============================================================================
# 1. CÀI ĐẶT THƯ VIỆN BÓC TÁCH DỮ LIỆU CHUYÊN SÂU
# =============================================================================
!pip install -q pdfplumber beautifulsoup4 requests tqdm vnstock
print("✅ Cài đặt môi trường bóc tách thành công!")

In [ ]:
# =============================================================================
# 2. KẾT NỐI GOOGLE DRIVE (SHARED FOLDER)
# =============================================================================
import os, sys, json, time, re, shutil
import urllib.request
from typing import Dict, List, Any, Optional
from google.colab import drive
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup
import pdfplumber

# Mount Google Drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/vnstock_data'
PDF_LAKE_DIR = os.path.join(BASE_DIR, 'pdf_lake')
os.makedirs(PDF_LAKE_DIR, exist_ok=True)

# Thư mục chứa PDF tạm thời trên ổ cứng 100GB của Colab
SCRATCH_DIR = '/content/temp_pdf_scratch'
os.makedirs(SCRATCH_DIR, exist_ok=True)

print(f"📁 Kho dữ liệu Google Drive: {PDF_LAKE_DIR}")
print(f"⚡ Thư mục stream tạm thời: {SCRATCH_DIR}")

In [ ]:
# =============================================================================
# 3. CẤU HÌNH WORKER & ĐỘ SÂU DỮ LIỆU CẦN LẤY
# =============================================================================
WORKER_ID = 0      # 0 cho Acc 1 | 1 cho Acc 2 | 2 cho Acc 3 | 3 cho Acc 4 | 4 cho Acc 5
TOTAL_WORKERS = 5

# Cấu hình số lượng báo cáo mỗi mã:
# - 4 BCTC: 4 quý gần nhất (~1 năm, đủ TTM & định giá nhanh)
# - 8 BCTC: 8 quý gần nhất (~2 năm, xem được xu hướng tăng trưởng)
# - 12 BCTC: 12 quý gần nhất (~3 năm)
MAX_BCTC_PER_SYMBOL = 8   # Tùy chỉnh: 4, 8 hoặc 12
MAX_BCTN_PER_SYMBOL = 2   # Số lượng Báo cáo thường niên gần nhất

SHARD_FILE = os.path.join(PDF_LAKE_DIR, f"bctc_shard_{WORKER_ID}.json")
EXISTING_LAKE_FILE = os.path.join(PDF_LAKE_DIR, "extracted_bctc_lake.json")

print(f"🚀 ĐANG CHẠY WORKER {WORKER_ID + 1}/{TOTAL_WORKERS}")
print(f"📊 Độ sâu lấy dữ liệu: Tối đa {MAX_BCTC_PER_SYMBOL} BCTC + {MAX_BCTN_PER_SYMBOL} BCTN mỗi mã")
print(f"💾 File kết quả đầu ra: {SHARD_FILE}")

In [ ]:
# =============================================================================
# 4. TẢI DANH MỤC CỔ PHIẾU & PHÂN CHIA NHIỆM VỤ (SHARDING)
# =============================================================================
symbols_cache_file = os.path.join(BASE_DIR, "all_symbols.json")
all_symbols = []

if os.path.exists(symbols_cache_file):
    try:
        with open(symbols_cache_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict):
                        s = item.get('symbol') or item.get('ticker')
                        if s:
                            all_symbols.append(s)
                    elif isinstance(item, str):
                        all_symbols.append(item)
    except Exception as e:
        print(f"Read symbols_cache_file error: {e}")

if not all_symbols:
    try:
        from vnstock import Listing
        lst = Listing()
        df = lst.all_symbols()
        if df is not None and not df.empty:
            col = 'symbol' if 'symbol' in df.columns else 'ticker'
            all_symbols = sorted(df[col].dropna().unique().tolist())
    except Exception as e:
        print(f"Fallback fetch symbols: {e}")

# Lọc danh sách mã hợp lệ (độ dài 3 ký tự chuẩn VN)
all_symbols = sorted(list(set([str(s).strip().upper() for s in all_symbols if s and len(str(s).strip()) == 3 and str(s).strip().isalnum()])))
print(f"📊 Toàn thị trường: {len(all_symbols)} mã")

# Phân chia danh sách cho worker hiện tại
my_symbols = [s for i, s in enumerate(all_symbols) if i % TOTAL_WORKERS == WORKER_ID]
print(f"🎯 Worker {WORKER_ID} được giao: {len(my_symbols)} mã")
print(f"Mẫu danh sách: {my_symbols[:8]}...")

In [ ]:
# =============================================================================
# 5. BCTC PARSER ENGINE (CHUẨN MỰC THÔNG TƯ 200 & BÁO CÁO THƯỜNG NIÊN)
# =============================================================================

def clean_num(val_str):
    if not val_str:
        return 0.0
    s = str(val_str).strip()
    is_neg = s.startswith('(') and s.endswith(')')
    s = re.sub(r'[^0-9,\.-]', '', s)
    if not s:
        return 0.0
    if '.' in s and ',' in s:
        if s.find('.') < s.find(','):
            s = s.replace('.', '').replace(',', '.')
        else:
            s = s.replace(',', '')
    elif ',' in s and '.' not in s:
        parts = s.split(',')
        if len(parts[-1]) == 3:
            s = s.replace(',', '')
        else:
            s = s.replace(',', '.')
    try:
        v = float(s)
        return -v if is_neg else v
    except:
        return 0.0

def detect_period(text):
    t = text.lower()
    quarter = None
    q_m = re.search(r'qu[ýy]\s*([1-4iv]+)', t)
    if q_m:
        q_raw = q_m.group(1).upper()
        mapping = {'1': 1, 'I': 1, '2': 2, 'II': 2, '3': 3, 'III': 3, '4': 4, 'IV': 4}
        quarter = mapping.get(q_raw)
    elif 'bán niên' in t or '6 tháng' in t:
        quarter = '6M'
    elif 'cả năm' in t or 'năm tài chính' in t:
        quarter = 'FY'
        
    year = None
    y_m = re.search(r'(?:năm|ktkt)\s*(20[12][0-9])', t)
    if y_m:
        year = int(y_m.group(1))
    is_audited = any(k in t for k in ['kiểm toán', 'đã kiểm toán', 'soát xét'])
    return {'quarter': quarter, 'year': year, 'is_audited': is_audited}

def extract_bctc_pdf(pdf_path):
    records = {}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = " ".join([page.extract_text() or "" for page in pdf.pages[:8]])
            period = detect_period(full_text)
            
            for page in pdf.pages:
                tables = page.extract_tables()
                for tbl in tables:
                    if not tbl:
                        continue
                    for row in tbl:
                        if not row or len(row) < 3:
                            continue
                        text_line = " ".join([str(c) for c in row if c]).lower()
                        if 'doanh thu thuần' in text_line or 'mã số 10' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val > 0 and 'revenue' not in records:
                                records['revenue'] = val
                        elif 'lợi nhuận sau thuế' in text_line or 'mã số 60' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val != 0 and 'net_profit' not in records:
                                records['net_profit'] = val
                        elif 'lưu chuyển tiền thuần từ hoạt động kinh doanh' in text_line or 'mã số 20' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val != 0 and 'cfo' not in records:
                                records['cfo'] = val
                        elif 'mua sắm' in text_line and ('tscđ' in text_line or 'tài sản cố định' in text_line):
                            val = abs(clean_num(row[-1] or (row[-2] if len(row) > 2 else 0)))
                            if val > 0 and 'capex' not in records:
                                records['capex'] = val
                        elif 'tổng cộng tài sản' in text_line or 'mã số 270' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val > 0 and 'total_assets' not in records:
                                records['total_assets'] = val
                        elif 'vốn chủ sở hữu' in text_line or 'mã số 400' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val > 0 and 'equity' not in records:
                                records['equity'] = val
                                
            if 'cfo' in records and 'capex' in records:
                records['fcf'] = records['cfo'] - records['capex']
            records['period'] = period
            return records
    except Exception as e:
        return {}

def extract_bctn_targets(pdf_path):
    targets = {}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages[:100]):
                t = (page.extract_text() or "").lower()
                if 'kế hoạch kinh doanh' in t or 'kế hoạch năm' in t:
                    rev_m = re.search(r'doanh\s*thu.*?([0-9]{1,3}(?:[\.,][0-9]{3})+|\d+)\s*(?:tỷ|triệu)?', t)
                    if rev_m and 'plan_revenue' not in targets:
                        targets['plan_revenue'] = rev_m.group(1)
                    prof_m = re.search(r'lợi\s*nhuận.*?([0-9]{1,3}(?:[\.,][0-9]{3})+|\d+)\s*(?:tỷ|triệu)?', t)
                    if prof_m and 'plan_profit' not in targets:
                        targets['plan_profit'] = prof_m.group(1)
                if 'cổ tức' in t:
                    div_m = re.search(r'cổ\s*tức.*?(\d{1,2}(?:[\.,]\d+)?\s*%)', t)
                    if div_m and 'expected_dividend' not in targets:
                        targets['expected_dividend'] = div_m.group(1)
                if len(targets) >= 3:
                    break
    except Exception:
        pass
    return targets

In [ ]:
# =============================================================================
# 6. CRAWLER LẤY TÀI LIỆU (CAFEF API)
# =============================================================================
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def get_symbol_pdf_links(symbol):
    """Lấy danh sách các tài liệu BCTC và Báo cáo thường niên gần nhất của mã"""
    url = f"https://s.cafef.vn/Ajax/Events_RelatedNews_New.aspx?symbol={symbol}&floorID=0&configID=0&PageIndex=1&PageSize=30&Type=2"
    docs = []
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a['href']
                title = a.get_text().strip()
                if href.lower().endswith('.pdf'):
                    full_url = href if href.startswith('http') else 'https://s.cafef.vn' + href
                    doc_type = 'BCTN' if ('thường niên' in title.lower() or 'annual' in title.lower()) else 'BCTC'
                    docs.append({'title': title, 'url': full_url, 'type': doc_type})
    except Exception as e:
        pass
    return docs

In [ ]:
# =============================================================================
# 7. VÒNG LẶP THI CÔNG STREAM & PURGE + KẾ THỪA CACHE CŨ (INCREMENTAL SYNC)
# =============================================================================
# 1. Nạp dữ liệu đã có từ shard hiện tại
shard_data = {}
if os.path.exists(SHARD_FILE):
    try:
        with open(SHARD_FILE, 'r', encoding='utf-8') as f:
            shard_data = json.load(f)
        print(f"🔄 Đã nạp {len(shard_data)} mã từ Checkpoint Shard hiện tại!")
    except Exception:
        shard_data = {}

# 2. Kế thừa dữ liệu từ extracted_bctc_lake.json cũ nếu có trên Drive
existing_lake = {}
if os.path.exists(EXISTING_LAKE_FILE):
    try:
        with open(EXISTING_LAKE_FILE, 'r', encoding='utf-8') as f:
            existing_lake = json.load(f)
        print(f"📚 Tìm thấy Data Lake cũ với {len(existing_lake)} mã. Sẽ kế thừa và không cào lại nếu đã đủ quý!")
    except Exception:
        existing_lake = {}

# 3. Lọc các mã thực sự cần chạy (chưa có hoặc chưa đủ số quý yêu cầu)
symbols_to_run = []
for sym in my_symbols:
    # Đã có trong shard hiện tại
    if sym in shard_data and len(shard_data[sym].get('periods', [])) >= MAX_BCTC_PER_SYMBOL:
        continue
    # Đã có trong lake cũ và đủ số quý
    if sym in existing_lake and len(existing_lake[sym].get('periods', [])) >= MAX_BCTC_PER_SYMBOL:
        shard_data[sym] = existing_lake[sym]  # Copy sang shard luôn để hoàn chỉnh
        continue
    symbols_to_run.append(sym)

print(f"⚡ Cần xử lý: {len(symbols_to_run)}/{len(my_symbols)} mã (Đã bỏ qua {len(my_symbols) - len(symbols_to_run)} mã có sẵn trong Cache)")

processed_count = 0
for sym in tqdm(symbols_to_run, desc=f"Worker-{WORKER_ID} Processing"):
    docs = get_symbol_pdf_links(sym)
    sym_result = shard_data.get(sym, {'symbol': sym, 'periods': [], 'annual_report_targets': {}, 'updated_at': int(time.time())})
    
    # Lấy tài liệu theo độ sâu đã cấu hình
    bctc_docs = [d for d in docs if d['type'] == 'BCTC'][:MAX_BCTC_PER_SYMBOL]
    bctn_docs = [d for d in docs if d['type'] == 'BCTN'][:MAX_BCTN_PER_SYMBOL]
    
    existing_titles = set([p.get('doc_title', '') for p in sym_result.get('periods', [])])
    
    # Bóc BCTC
    for d in bctc_docs:
        if d['title'] in existing_titles:
            continue  # Đã có quý này, bỏ qua
            
        temp_pdf = os.path.join(SCRATCH_DIR, f"{sym}_bctc_temp.pdf")
        try:
            r = requests.get(d['url'], headers=HEADERS, timeout=20)
            if r.status_code == 200:
                with open(temp_pdf, 'wb') as f:
                    f.write(r.content)
                parsed = extract_bctc_pdf(temp_pdf)
                if parsed:
                    parsed['doc_title'] = d['title']
                    sym_result['periods'].append(parsed)
        except Exception:
            pass
        finally:
            if os.path.exists(temp_pdf):
                os.remove(temp_pdf)  # Xoá ngay PDF thô
                
    # Bóc Báo cáo thường niên (nếu chưa có)
    if not sym_result.get('annual_report_targets'):
        for d in bctn_docs:
            temp_pdf = os.path.join(SCRATCH_DIR, f"{sym}_bctn_temp.pdf")
            try:
                r = requests.get(d['url'], headers=HEADERS, timeout=30)
                if r.status_code == 200:
                    with open(temp_pdf, 'wb') as f:
                        f.write(r.content)
                    targets = extract_bctn_targets(temp_pdf)
                    if targets:
                        sym_result['annual_report_targets'] = targets
                        break
            except Exception:
                pass
            finally:
                if os.path.exists(temp_pdf):
                    os.remove(temp_pdf)  # Xoá ngay PDF thô
                    
    shard_data[sym] = sym_result
    processed_count += 1
    
    # Lưu Checkpoint mỗi 5 mã
    if processed_count % 5 == 0 or sym == symbols_to_run[-1]:
        tmp_save = SHARD_FILE + ".tmp"
        with open(tmp_save, 'w', encoding='utf-8') as f:
            json.dump(shard_data, f, ensure_ascii=False, indent=2)
        os.replace(tmp_save, SHARD_FILE)
        
    time.sleep(0.2)  # Tối ưu tốc độ

print(f"🎉 CHÚC MỪNG: Worker {WORKER_ID} đã hoàn thành xuất sắc toàn bộ {len(my_symbols)} mã!")
print(f"File kết quả đã an toàn tại: {SHARD_FILE}")